# Multi-Asset CTA Strategy V2 — Transition Strategy

## 04 — Transition Classification and Incremental Feature Testing

Book 04 is the first formal predictive/falsification stage of V2.

Book 02 defined fast countertrend candidates and labelled them ex post as **genuine** or **failed** according to whether the slow conventional trend subsequently confirmed the reversal. Book 03 constructed only information observable at the candidate date.

Book 04 asks:

> **Can contemporaneous transition information predict which candidates become genuine slow-trend transitions, out of sample?**

It also asks the more demanding falsification question:

> **Does MACD, SuperbCommand, volatility state, or nonlinear modelling add information beyond conventional transition geometry?**

### Main model ladder

Two complementary tests are used.

**Transparent logistic ladder**
1. Conventional geometry
2. + MACD
3. + SuperbCommand
4. + volatility

**Nonlinear Random Forest ablation**
1. RF conventional geometry
2. RF + MACD
3. RF + MACD + SuperbCommand
4. RF + MACD + SuperbCommand + volatility
5. **RF conventional + SuperbCommand, with MACD removed**

The fifth specification is the final targeted falsification test motivated by the first Book 04 results: MACD weakened the conventional baseline, while SuperbCommand added substantial nonlinear information. It tests whether SuperbCommand is cleaner and stronger when MACD is excluded.

The second ladder is crucial. It tells us whether a strong nonlinear model is merely exploiting nonlinearities already present in conventional trend geometry, or whether the richer transition features genuinely add information.

### Probability treatment

Balanced class weights are useful for ranking and discrimination, but they alter the effective class prior and therefore raw probability calibration.

Book 04 therefore reports:

- **balanced logistic/RF models** for discrimination and ranking;
- **unweighted full logistic/RF models** for more interpretable probability calibration.

The project ultimately wants a useful estimate of:

\[
P(\text{genuine transition}\mid I_t)
\]

because later exposure sizing may depend on that probability.

### Validation principles

- chronological expanding-window testing;
- no random train/test split;
- no future-return features;
- no confirmation date or lead-time information;
- no synthetic resampling;
- identical annual folds across model variants;
- results overall, by asset class, by transition direction, and with digital assets excluded;
- temporal stability reported explicitly.

### Primary metrics

- ROC AUC;
- PR AUC / Average Precision;
- Brier score;
- log loss;
- top-quintile genuine-transition rate and lift;
- calibration error;
- year-by-year stability.

### Divergence geometry

Book 03's canonical divergence representation is preserved:

`t1 → t2 → candidate`

where:

- `t1` = previous same-direction crossover;
- `t2` = most recent same-direction crossover;
- `candidate` = Book 02 transition candidate date.

Continuous **anchor spacing** (`t2 - t1`) and **divergence recency** (`candidate - t2`) are the canonical timing variables. Fixed 4/12-week and 21/63-observation windows are descriptive features only, not validity rules.

### Scope

This book tests **predictive information**. It does not yet decide trading exposure, portfolio weights, or whether the transition sleeve improves the combined CTA. Those belong later.


In [ ]:
# =========================================================
# 1) INSTALLS, IMPORTS, DRIVE, PATHS
# =========================================================
!pip -q install pyarrow scikit-learn

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

warnings.filterwarnings("ignore")

PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2")
V203 = PROJECT / "v2.03"
V204 = PROJECT / "v2.04"

for p in [V204 / "config", V204 / "data", V204 / "results"]:
    p.mkdir(parents=True, exist_ok=True)

FEATURES_PARQUET = V203 / "data" / "v2_03_candidate_features.parquet"
FEATURES_CSV = V203 / "results" / "v2_03_candidate_features.csv"

CONFIG = {
    "book": "V2.04",
    "research_end": "2025-12-31",
    "minimum_train_years": 8,
    "first_test_year": 2008,
    "top_quantile": 0.20,
    "calibration_bins": 10,
    "logistic_C": 1.0,
    "rf_estimators": 500,
    "rf_min_samples_leaf": 20,
    "random_state": 42,
}

print("Project:", PROJECT)
print("Book 04 outputs:", V204)
print(json.dumps(CONFIG, indent=2))


Mounted at /content/drive
Project: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2
Book 04 outputs: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.04
{
  "book": "V2.04",
  "research_end": "2025-12-31",
  "minimum_train_years": 8,
  "first_test_year": 2008,
  "top_quantile": 0.2,
  "calibration_bins": 10,
  "logistic_C": 1.0,
  "rf_estimators": 500,
  "rf_min_samples_leaf": 20,
  "random_state": 42
}


In [ ]:
# =========================================================
# 2) LOAD BOOK 03 CANDIDATE FEATURES
# =========================================================
if FEATURES_PARQUET.exists():
    features = pd.read_parquet(FEATURES_PARQUET)
    source_path = FEATURES_PARQUET
elif FEATURES_CSV.exists():
    features = pd.read_csv(FEATURES_CSV)
    source_path = FEATURES_CSV
else:
    raise FileNotFoundError(
        "Book 03 candidate features not found.\n"
        f"Expected either:\n  {FEATURES_PARQUET}\n  {FEATURES_CSV}"
    )

features["candidate_date"] = pd.to_datetime(features["candidate_date"])
features = features.sort_values(["candidate_date", "market"]).reset_index(drop=True)

allowed_labels = {"genuine", "failed"}
observed_labels = set(features["label"].dropna().unique())
if not observed_labels.issubset(allowed_labels):
    raise ValueError(f"Unexpected labels: {observed_labels}")

features["target"] = (features["label"] == "genuine").astype(int)
features["candidate_direction"] = pd.to_numeric(
    features["candidate_direction"], errors="coerce"
)
features["year"] = features["candidate_date"].dt.year

print("Loaded:", source_path)
print("Rows:", len(features))
print("Markets:", features["market"].nunique())
print("Columns:", len(features.columns))
print("Date range:", features["candidate_date"].min(), "to", features["candidate_date"].max())
print("Genuine rate:", round(features["target"].mean(), 4))
print()
print(
    features.groupby("category")
    .agg(events=("target", "size"), genuine_rate=("target", "mean"))
    .sort_values("events", ascending=False)
)


Loaded: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.03/data/v2_03_candidate_features.parquet
Rows: 3462
Markets: 53
Columns: 155
Date range: 2000-01-19 00:00:00 to 2025-12-31 00:00:00
Genuine rate: 0.3186

                events  genuine_rate
category                            
INDICES           1176      0.275510
COMMODITIES       1122      0.338681
FX                 717      0.362622
BONDS_RATES        416      0.300481
DIGITAL_ASSETS      31      0.451613


## Feature governance and leakage protection

Feature families are specified explicitly. Book 03 contains ex-post labels and forward returns that must never enter the predictor matrix.

The **conventional** family is the core falsification benchmark. If richer features cannot beat it out of sample, they do not earn inclusion merely because they are conceptually interesting.


In [ ]:
# =========================================================
# 3) FEATURE FAMILIES + LEAKAGE GUARD
# =========================================================

CONVENTIONAL = [
    "tsmom_distance_pct",
    "ma_spread_pct",
    "ma_fast_slope_21",
    "ma_slow_slope_21",
    "breakout_mid_distance_pct",
    "channel_position",
    "slow_vote_sum",
    "slow_vote_abs",
    "fast_slow_disagreement",
    "fast_countertrend_strength",
    "established_trend_age",
    "incumbent_cumulative_move",
    "drawdown_from_126_high",
    "recovery_from_126_low",
    "adverse_move_vs_incumbent",
    "candidate_dir_tsmom_distance",
    "candidate_dir_ma_spread",
    "candidate_dir_breakout_distance",
]

MACD = [
    "macd_pct",
    "macd_signal_pct",
    "macd_hist_pct",
    "macd_hist_z",
    "macd_cross_up",
    "macd_cross_down",
    "candidate_divergence_flag",
    "candidate_divergence_strength",
    "candidate_divergence_age",
    "candidate_divergence_anchor_gap",
    "candidate_divergence_active_21",
    "candidate_divergence_active_63",
]

SUPERBCOMMAND = [
    "sc_candidate_dir_osc_pct",
    "sc_candidate_dir_signal_pct",
    "sc_candidate_dir_spread_pct",
    "sc_candidate_dir_osc_slope_pct",
    "sc_candidate_dir_signal_slope_pct",
    "sc_osc_rising",
    "sc_osc_falling",
    "sc_signal_rising",
    "sc_signal_falling",
    "sc_osc_turns_red",
    "sc_bb_width_pct",
    "sc_osc_minus_upper_bb_pct",
    "sc_osc_minus_lower_bb_pct",
    "sc_osc_above_upper_bb",
    "sc_osc_below_lower_bb",
    "sc_osc_between_bbs",
    "sc_signal_cross_above_zero",
    "sc_osc_cross_above_signal",
    "sc_osc_cross_below_signal",
    "sc_osc_cross_above_lower_bb",
    "sc_osc_cross_below_upper_bb",
    "sc_candidate_supertrend_alignment",
    "sc_supertrend_flip_bull",
    "sc_supertrend_flip_bear",
    "sc_candidate_price_vs_supertrend_atr",
    "sc_standby_anchor_condition",
    "sc_early_reversal_intersection_raw",
    "sc_early_reversal_setup_bull",
    "sc_pullback_setup_bull",
    "sc_profit_taking_turn_bull",
    "sc_candidate_divergence_flag",
    "sc_candidate_divergence_strength",
    "sc_candidate_divergence_age_weeks",
    "sc_candidate_divergence_anchor_gap_weeks",
    "sc_candidate_divergence_active_4w",
    "sc_candidate_divergence_active_12w",
    "sc_candidate_alignment_score",
]

# Volatility is deliberately held out of the prior SC layer so its
# incremental contribution can be tested separately.
VOLATILITY = [
    "sc_atr_pct",
    "sc_vol_cluster",
    "sc_vol_cluster_change",
    "sc_assigned_atr_centroid",
    "sc_high_vol_centroid",
    "sc_mid_vol_centroid",
    "sc_low_vol_centroid",
]

MODEL_FEATURES = {
    "01_conventional": CONVENTIONAL,
    "02_plus_macd": CONVENTIONAL + MACD,
    "03_plus_superbcommand": CONVENTIONAL + MACD + SUPERBCOMMAND,
    "04_plus_volatility": CONVENTIONAL + MACD + SUPERBCOMMAND + VOLATILITY,

    # Decisive final ablation: does SuperbCommand work better without
    # the MACD block that has so far weakened the conventional baseline?
    "05_conventional_plus_superbcommand_no_macd": CONVENTIONAL + SUPERBCOMMAND,
}

MODEL_FEATURES = {
    name: list(dict.fromkeys(cols))
    for name, cols in MODEL_FEATURES.items()
}

missing = {
    name: [c for c in cols if c not in features.columns]
    for name, cols in MODEL_FEATURES.items()
}
missing = {k: v for k, v in missing.items() if v}

if missing:
    raise ValueError(f"Expected Book 03 predictors are missing:\n{missing}")

FORBIDDEN_EXACT = {
    "label",
    "target",
    "confirmation_date",
    "book02_confirmation_date",
    "lead_time_trading_days",
    "label_horizon_end",
}

FORBIDDEN_PATTERNS = [
    "oriented_return",
    "forward_return",
    "ret_21",
    "ret_63",
    "ret_126",
    "ret_252",
    "confirmation_date",
    "lead_time",
]

selected = sorted(set(sum(MODEL_FEATURES.values(), [])))
leaks = [
    c for c in selected
    if c in FORBIDDEN_EXACT or any(p in c for p in FORBIDDEN_PATTERNS)
]

if leaks:
    raise ValueError(f"LEAKAGE GUARD FAILED: {leaks}")

print("Leakage guard: PASSED")
for name, cols in MODEL_FEATURES.items():
    print(f"{name}: {len(cols)} predictors")


Leakage guard: PASSED
01_conventional: 18 predictors
02_plus_macd: 30 predictors
03_plus_superbcommand: 67 predictors
04_plus_volatility: 74 predictors
05_conventional_plus_superbcommand_no_macd: 55 predictors


In [ ]:
# =========================================================
# 4) EXPANDING-WINDOW ANNUAL FOLDS
# =========================================================
first_year = int(features["year"].min())
last_year = int(features["year"].max())

start_test_year = max(
    CONFIG["first_test_year"],
    first_year + CONFIG["minimum_train_years"],
)

test_years = [
    y for y in range(start_test_year, last_year + 1)
    if (features["year"] == y).any()
]

folds = []
for y in test_years:
    train_idx = features.index[features["year"] < y].to_numpy()
    test_idx = features.index[features["year"] == y].to_numpy()

    if len(train_idx) == 0 or len(test_idx) == 0:
        continue
    if features.loc[train_idx, "target"].nunique() < 2:
        continue

    folds.append((y, train_idx, test_idx))

if not folds:
    raise RuntimeError("No valid expanding-window folds created.")

fold_audit = pd.DataFrame([
    {
        "test_year": y,
        "train_events": len(tr),
        "test_events": len(te),
        "train_start": features.loc[tr, "candidate_date"].min(),
        "train_end": features.loc[tr, "candidate_date"].max(),
        "test_start": features.loc[te, "candidate_date"].min(),
        "test_end": features.loc[te, "candidate_date"].max(),
        "train_genuine_rate": features.loc[tr, "target"].mean(),
        "test_genuine_rate": features.loc[te, "target"].mean(),
    }
    for y, tr, te in folds
])

print(fold_audit.to_string(index=False))


 test_year  train_events  test_events train_start  train_end test_start   test_end  train_genuine_rate  test_genuine_rate
      2008           775          118  2000-01-19 2007-12-31 2008-01-03 2008-12-16            0.295484           0.372881
      2009           893          147  2000-01-19 2008-12-16 2009-01-05 2009-12-31            0.305711           0.394558
      2010          1040          136  2000-01-19 2009-12-31 2010-01-04 2010-12-27            0.318269           0.147059
      2011          1176          147  2000-01-19 2010-12-27 2011-01-07 2011-12-30            0.298469           0.367347
      2012          1323          165  2000-01-19 2011-12-30 2012-01-05 2012-12-28            0.306122           0.345455
      2013          1488          130  2000-01-19 2012-12-28 2013-01-03 2013-12-30            0.310484           0.376923
      2014          1618          147  2000-01-19 2013-12-30 2014-01-02 2014-12-31            0.315822           0.285714
      2015          1765

In [ ]:
# =========================================================
# 5) MODEL FACTORIES + GENERIC OOS PREDICTION ENGINE
# =========================================================
def make_logistic(class_weight="balanced"):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            C=CONFIG["logistic_C"],
            max_iter=5000,
            class_weight=class_weight,
            random_state=CONFIG["random_state"],
        )),
    ])

def make_rf(class_weight="balanced_subsample"):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", RandomForestClassifier(
            n_estimators=CONFIG["rf_estimators"],
            min_samples_leaf=CONFIG["rf_min_samples_leaf"],
            max_features="sqrt",
            class_weight=class_weight,
            n_jobs=-1,
            random_state=CONFIG["random_state"],
        )),
    ])

def expanding_predictions(model_name, feature_cols, estimator):
    rows = []

    for test_year, train_idx, test_idx in folds:
        X_train = (
            features.loc[train_idx, feature_cols]
            .replace([np.inf, -np.inf], np.nan)
        )
        y_train = features.loc[train_idx, "target"]

        X_test = (
            features.loc[test_idx, feature_cols]
            .replace([np.inf, -np.inf], np.nan)
        )

        est = clone(estimator)
        est.fit(X_train, y_train)
        p = est.predict_proba(X_test)[:, 1]

        part = features.loc[test_idx, [
            "candidate_date",
            "market",
            "category",
            "candidate_direction",
            "label",
            "target",
        ]].copy()

        part["test_year"] = test_year
        part["model"] = model_name
        part["prob_genuine"] = p
        rows.append(part)

    return pd.concat(rows, ignore_index=True)

def expanding_base_rate():
    rows = []

    for test_year, train_idx, test_idx in folds:
        p0 = features.loc[train_idx, "target"].mean()

        part = features.loc[test_idx, [
            "candidate_date",
            "market",
            "category",
            "candidate_direction",
            "label",
            "target",
        ]].copy()

        part["test_year"] = test_year
        part["model"] = "00_base_rate"
        part["prob_genuine"] = p0
        rows.append(part)

    return pd.concat(rows, ignore_index=True)

print("Model factories ready.")


Model factories ready.


In [ ]:
# =========================================================
# 6) BALANCED TRANSPARENT LOGISTIC LADDER
# =========================================================
balanced_parts = [expanding_base_rate()]

for name, cols in MODEL_FEATURES.items():
    print("Fitting balanced logistic:", name)
    balanced_parts.append(
        expanding_predictions(
            model_name=name,
            feature_cols=cols,
            estimator=make_logistic(class_weight="balanced"),
        )
    )

balanced_predictions = pd.concat(balanced_parts, ignore_index=True)

print()
print("Balanced logistic prediction rows:", len(balanced_predictions))
print(balanced_predictions.groupby("model").size())


Fitting balanced logistic: 01_conventional
Fitting balanced logistic: 02_plus_macd
Fitting balanced logistic: 03_plus_superbcommand
Fitting balanced logistic: 04_plus_volatility
Fitting balanced logistic: 05_conventional_plus_superbcommand_no_macd

Balanced logistic prediction rows: 16122
model
00_base_rate                                  2687
01_conventional                               2687
02_plus_macd                                  2687
03_plus_superbcommand                         2687
04_plus_volatility                            2687
05_conventional_plus_superbcommand_no_macd    2687
dtype: int64


In [ ]:
# =========================================================
# 7) BALANCED RANDOM FOREST FEATURE-FAMILY ABLATION
# =========================================================
RF_FEATURES = {
    "RF_01_conventional": MODEL_FEATURES["01_conventional"],
    "RF_02_plus_macd": MODEL_FEATURES["02_plus_macd"],
    "RF_03_plus_superbcommand": MODEL_FEATURES["03_plus_superbcommand"],
    "RF_04_plus_volatility": MODEL_FEATURES["04_plus_volatility"],

    # Direct falsification test:
    # conventional geometry + SuperbCommand, with MACD removed.
    "RF_05_conventional_plus_superbcommand_no_macd":
        MODEL_FEATURES["05_conventional_plus_superbcommand_no_macd"],
}

rf_parts = []

for name, cols in RF_FEATURES.items():
    print("Fitting balanced RF:", name)
    rf_parts.append(
        expanding_predictions(
            model_name=name,
            feature_cols=cols,
            estimator=make_rf(class_weight="balanced_subsample"),
        )
    )

rf_predictions = pd.concat(rf_parts, ignore_index=True)

print()
print("RF ablation prediction rows:", len(rf_predictions))
print(rf_predictions.groupby("model").size())


Fitting balanced RF: RF_01_conventional
Fitting balanced RF: RF_02_plus_macd
Fitting balanced RF: RF_03_plus_superbcommand
Fitting balanced RF: RF_04_plus_volatility
Fitting balanced RF: RF_05_conventional_plus_superbcommand_no_macd

RF ablation prediction rows: 13435
model
RF_01_conventional                               2687
RF_02_plus_macd                                  2687
RF_03_plus_superbcommand                         2687
RF_04_plus_volatility                            2687
RF_05_conventional_plus_superbcommand_no_macd    2687
dtype: int64


In [ ]:
# =========================================================
# 8) UNWEIGHTED MODELS FOR PROBABILITY CALIBRATION
# =========================================================
full_cols = MODEL_FEATURES["04_plus_volatility"]
no_macd_cols = MODEL_FEATURES["05_conventional_plus_superbcommand_no_macd"]

print("Fitting unweighted full logistic...")
logit_unweighted = expanding_predictions(
    model_name="LOGIT_unweighted_full",
    feature_cols=full_cols,
    estimator=make_logistic(class_weight=None),
)

print("Fitting unweighted full Random Forest...")
rf_unweighted = expanding_predictions(
    model_name="RF_unweighted_full",
    feature_cols=full_cols,
    estimator=make_rf(class_weight=None),
)

print("Fitting unweighted RF conventional + SuperbCommand (no MACD)...")
rf_no_macd_unweighted = expanding_predictions(
    model_name="RF_unweighted_C_plus_SC_no_MACD",
    feature_cols=no_macd_cols,
    estimator=make_rf(class_weight=None),
)

unweighted_predictions = pd.concat(
    [logit_unweighted, rf_unweighted, rf_no_macd_unweighted],
    ignore_index=True,
)

print()
print(unweighted_predictions.groupby("model").size())


Fitting unweighted full logistic...
Fitting unweighted full Random Forest...
Fitting unweighted RF conventional + SuperbCommand (no MACD)...

model
LOGIT_unweighted_full              2687
RF_unweighted_C_plus_SC_no_MACD    2687
RF_unweighted_full                 2687
dtype: int64


In [ ]:
# =========================================================
# 9) METRICS
# =========================================================
def safe_auc(y, p):
    y = pd.Series(y)
    if y.nunique() < 2:
        return np.nan
    return roc_auc_score(y, p)

def safe_pr_auc(y, p):
    y = pd.Series(y)
    if y.nunique() < 2:
        return np.nan
    return average_precision_score(y, p)

def safe_logloss(y, p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return log_loss(y, p, labels=[0, 1])

def metric_row(g):
    y = g["target"].astype(int).to_numpy()
    p = g["prob_genuine"].astype(float).to_numpy()

    base_rate = float(np.mean(y))

    # Top bucket is formed within each OOS test year so that a shift in
    # calibration level across years cannot dominate the ranking statistic.
    temp = g.copy()
    temp["pct_rank"] = temp.groupby("test_year")["prob_genuine"].rank(
        method="average",
        pct=True,
    )

    top = temp[
        temp["pct_rank"] >= 1 - CONFIG["top_quantile"]
    ]

    top_rate = top["target"].mean() if len(top) else np.nan

    return pd.Series({
        "events": len(g),
        "genuine_rate": base_rate,
        "roc_auc": safe_auc(y, p),
        "pr_auc": safe_pr_auc(y, p),
        "brier": brier_score_loss(y, p),
        "log_loss": safe_logloss(y, p),
        "mean_predicted_probability": float(np.mean(p)),
        "top_quintile_events": len(top),
        "top_quintile_genuine_rate": top_rate,
        "top_quintile_lift": (
            top_rate / base_rate
            if base_rate > 0 and pd.notna(top_rate)
            else np.nan
        ),
    })

def metrics_overall(pred_df):
    return (
        pred_df.groupby("model", group_keys=False)
        .apply(metric_row)
        .reset_index()
    )

def metrics_by_category(pred_df):
    return (
        pred_df.groupby(["model", "category"], group_keys=False)
        .apply(metric_row)
        .reset_index()
    )

def metrics_by_direction(pred_df):
    x = pred_df.copy()
    x["direction_name"] = np.where(
        x["candidate_direction"] > 0,
        "BEAR_TO_BULL",
        "BULL_TO_BEAR",
    )
    return (
        x.groupby(["model", "direction_name"], group_keys=False)
        .apply(metric_row)
        .reset_index()
    )

balanced_metrics = metrics_overall(balanced_predictions)
rf_ablation_metrics = metrics_overall(rf_predictions)
unweighted_metrics = metrics_overall(unweighted_predictions)

print("BALANCED LOGISTIC LADDER")
print(balanced_metrics.to_string(index=False))

print("\nBALANCED RF ABLATION")
print(rf_ablation_metrics.to_string(index=False))

print("\nUNWEIGHTED FULL MODELS")
print(unweighted_metrics.to_string(index=False))


BALANCED LOGISTIC LADDER
                                     model  events  genuine_rate  roc_auc   pr_auc    brier  log_loss  mean_predicted_probability  top_quintile_events  top_quintile_genuine_rate  top_quintile_lift
                              00_base_rate  2687.0       0.32527 0.452015 0.296840 0.220358  0.632837                    0.318019                  0.0                        NaN                NaN
                           01_conventional  2687.0       0.32527 0.709633 0.528194 0.218185  0.633869                    0.476532                547.0                   0.542962           1.669265
                              02_plus_macd  2687.0       0.32527 0.704679 0.529417 0.219200  0.635684                    0.475339                547.0                   0.555759           1.708608
                     03_plus_superbcommand  2687.0       0.32527 0.710522 0.533592 0.215891  0.626237                    0.463942                547.0                   0.553931          

In [ ]:
# =========================================================
# 10) INCREMENTAL / ABLATION TABLES
# =========================================================
logit_order = [
    "00_base_rate",
    "01_conventional",
    "02_plus_macd",
    "03_plus_superbcommand",
    "04_plus_volatility",
    "05_conventional_plus_superbcommand_no_macd",
]

balanced_incremental = (
    balanced_metrics.set_index("model")
    .reindex(logit_order)
    .reset_index()
)

rf_order = [
    "RF_01_conventional",
    "RF_02_plus_macd",
    "RF_03_plus_superbcommand",
    "RF_04_plus_volatility",
    "RF_05_conventional_plus_superbcommand_no_macd",
]

rf_incremental = (
    rf_ablation_metrics.set_index("model")
    .reindex(rf_order)
    .reset_index()
)

for table in [balanced_incremental, rf_incremental]:
    for col in [
        "roc_auc",
        "pr_auc",
        "brier",
        "log_loss",
        "top_quintile_lift",
    ]:
        table[f"delta_vs_previous_{col}"] = table[col].diff()

# Direct comparisons for the final no-MACD test.
def direct_comparison(metrics_df, challenger, benchmark, label):
    m = metrics_df.set_index("model")
    rows = []
    for metric in [
        "roc_auc", "pr_auc", "brier", "log_loss",
        "top_quintile_genuine_rate", "top_quintile_lift"
    ]:
        challenger_value = m.loc[challenger, metric]
        benchmark_value = m.loc[benchmark, metric]
        rows.append({
            "comparison": label,
            "metric": metric,
            "challenger": challenger,
            "benchmark": benchmark,
            "challenger_value": challenger_value,
            "benchmark_value": benchmark_value,
            "difference_challenger_minus_benchmark":
                challenger_value - benchmark_value,
        })
    return pd.DataFrame(rows)

no_macd_comparison = pd.concat([
    direct_comparison(
        rf_ablation_metrics,
        "RF_05_conventional_plus_superbcommand_no_macd",
        "RF_01_conventional",
        "C+SC(no MACD) vs C",
    ),
    direct_comparison(
        rf_ablation_metrics,
        "RF_05_conventional_plus_superbcommand_no_macd",
        "RF_03_plus_superbcommand",
        "C+SC(no MACD) vs C+MACD+SC",
    ),
    direct_comparison(
        balanced_metrics,
        "05_conventional_plus_superbcommand_no_macd",
        "01_conventional",
        "LOGIT C+SC(no MACD) vs C",
    ),
    direct_comparison(
        balanced_metrics,
        "05_conventional_plus_superbcommand_no_macd",
        "03_plus_superbcommand",
        "LOGIT C+SC(no MACD) vs C+MACD+SC",
    ),
], ignore_index=True)

print("BALANCED LOGISTIC ABLATION")
print(balanced_incremental.to_string(index=False))

print("\nRANDOM FOREST FEATURE-FAMILY ABLATION")
print(rf_incremental.to_string(index=False))

print("\nDECISIVE NO-MACD COMPARISONS")
print(no_macd_comparison.to_string(index=False))


BALANCED LOGISTIC ABLATION
                                     model  events  genuine_rate  roc_auc   pr_auc    brier  log_loss  mean_predicted_probability  top_quintile_events  top_quintile_genuine_rate  top_quintile_lift  delta_vs_previous_roc_auc  delta_vs_previous_pr_auc  delta_vs_previous_brier  delta_vs_previous_log_loss  delta_vs_previous_top_quintile_lift
                              00_base_rate  2687.0       0.32527 0.452015 0.296840 0.220358  0.632837                    0.318019                  0.0                        NaN                NaN                        NaN                       NaN                      NaN                         NaN                                  NaN
                           01_conventional  2687.0       0.32527 0.709633 0.528194 0.218185  0.633869                    0.476532                547.0                   0.542962           1.669265                   0.257617                  0.231355                -0.002173                   

In [ ]:
# =========================================================
# 11) SUBGROUP ROBUSTNESS
# =========================================================
# Combine the two balanced ladders for consistent subgroup comparison.
all_balanced = pd.concat(
    [balanced_predictions, rf_predictions],
    ignore_index=True,
)

metrics_category = metrics_by_category(all_balanced)
metrics_direction = metrics_by_direction(all_balanced)

traditional_balanced = all_balanced[
    all_balanced["category"] != "DIGITAL_ASSETS"
].copy()
traditional_metrics = metrics_overall(traditional_balanced)

print("BY DIRECTION")
print(metrics_direction.to_string(index=False))

print("\nTRADITIONAL ASSETS ONLY")
print(traditional_metrics.to_string(index=False))


BY DIRECTION
                                        model direction_name  events  genuine_rate  roc_auc   pr_auc    brier  log_loss  mean_predicted_probability  top_quintile_events  top_quintile_genuine_rate  top_quintile_lift
                                 00_base_rate   BEAR_TO_BULL  1204.0      0.351329 0.445478 0.317116 0.229807  0.652632                    0.318340                  0.0                        NaN                NaN
                                 00_base_rate   BULL_TO_BEAR  1483.0      0.304113 0.456694 0.284965 0.212686  0.616766                    0.317757                  0.0                        NaN                NaN
                              01_conventional   BEAR_TO_BULL  1204.0      0.351329 0.750417 0.606509 0.210831  0.613732                    0.491143                250.0                   0.644000           1.833040
                              01_conventional   BULL_TO_BEAR  1483.0      0.304113 0.669726 0.454073 0.224155  0.650217        

In [ ]:
# =========================================================
# 12) TEMPORAL STABILITY
# =========================================================
stability_source = pd.concat(
    [
        balanced_predictions,
        rf_predictions,
        unweighted_predictions,
    ],
    ignore_index=True,
)

yearly_metrics = (
    stability_source.groupby(["model", "test_year"], group_keys=False)
    .apply(metric_row)
    .reset_index()
)

yearly_stability = (
    yearly_metrics.groupby("model")
    .agg(
        test_years=("test_year", "nunique"),
        mean_yearly_auc=("roc_auc", "mean"),
        median_yearly_auc=("roc_auc", "median"),
        pct_years_auc_above_050=(
            "roc_auc",
            lambda x: np.mean(pd.Series(x).dropna() > 0.50),
        ),
        mean_yearly_pr_auc=("pr_auc", "mean"),
        mean_yearly_brier=("brier", "mean"),
        mean_yearly_log_loss=("log_loss", "mean"),
    )
    .reset_index()
)

print(yearly_stability.to_string(index=False))


                                        model  test_years  mean_yearly_auc  median_yearly_auc  pct_years_auc_above_050  mean_yearly_pr_auc  mean_yearly_brier  mean_yearly_log_loss
                                 00_base_rate          18         0.500000           0.500000                      0.0            0.322250           0.219270              0.630562
                              01_conventional          18         0.711528           0.698354                      1.0            0.533881           0.217714              0.632512
                                 02_plus_macd          18         0.708126           0.697667                      1.0            0.539864           0.218638              0.634133
                        03_plus_superbcommand          18         0.710915           0.718959                      1.0            0.541455           0.215644              0.625122
                           04_plus_volatility          18         0.708970           0.716887       

In [ ]:
# =========================================================
# 13) CALIBRATION OF UNWEIGHTED FULL MODELS
# =========================================================
def make_calibration_table(pred_df, n_bins=10):
    rows = []

    for model, g in pred_df.groupby("model"):
        x = g.copy()
        x["prob_bin"] = pd.qcut(
            x["prob_genuine"],
            q=n_bins,
            labels=False,
            duplicates="drop",
        )

        table = (
            x.groupby("prob_bin")
            .agg(
                events=("target", "size"),
                mean_predicted_probability=("prob_genuine", "mean"),
                observed_genuine_rate=("target", "mean"),
            )
            .reset_index()
        )
        table.insert(0, "model", model)
        rows.append(table)

    return pd.concat(rows, ignore_index=True)

def calibration_error_summary(pred_df, n_bins=10):
    rows = []

    for model, g in pred_df.groupby("model"):
        x = g.copy()
        x["prob_bin"] = pd.qcut(
            x["prob_genuine"],
            q=n_bins,
            labels=False,
            duplicates="drop",
        )

        bins = (
            x.groupby("prob_bin")
            .agg(
                events=("target", "size"),
                mean_pred=("prob_genuine", "mean"),
                observed=("target", "mean"),
            )
            .reset_index()
        )

        bins["abs_error"] = (bins["mean_pred"] - bins["observed"]).abs()
        bins["weight"] = bins["events"] / bins["events"].sum()
        ece = (bins["abs_error"] * bins["weight"]).sum()

        rows.append({
            "model": model,
            "events": len(x),
            "actual_genuine_rate": x["target"].mean(),
            "mean_predicted_probability": x["prob_genuine"].mean(),
            "median_predicted_probability": x["prob_genuine"].median(),
            "ece_10bin": ece,
            "brier": brier_score_loss(
                x["target"],
                x["prob_genuine"],
            ),
            "log_loss": safe_logloss(
                x["target"],
                x["prob_genuine"],
            ),
            "roc_auc": safe_auc(
                x["target"],
                x["prob_genuine"],
            ),
            "pr_auc": safe_pr_auc(
                x["target"],
                x["prob_genuine"],
            ),
        })

    return pd.DataFrame(rows)

unweighted_calibration = make_calibration_table(
    unweighted_predictions,
    n_bins=CONFIG["calibration_bins"],
)

unweighted_calibration_error = calibration_error_summary(
    unweighted_predictions,
    n_bins=CONFIG["calibration_bins"],
)

print("UNWEIGHTED CALIBRATION ERROR")
print(unweighted_calibration_error.to_string(index=False))

print("\nCALIBRATION TABLE")
print(unweighted_calibration.to_string(index=False))


UNWEIGHTED CALIBRATION ERROR
                          model  events  actual_genuine_rate  mean_predicted_probability  median_predicted_probability  ece_10bin    brier  log_loss  roc_auc   pr_auc
          LOGIT_unweighted_full    2687              0.32527                    0.315190                      0.287493   0.038391 0.196924  0.587118 0.702956 0.524159
RF_unweighted_C_plus_SC_no_MACD    2687              0.32527                    0.324119                      0.317591   0.047371 0.184794  0.550082 0.747684 0.576291
             RF_unweighted_full    2687              0.32527                    0.327096                      0.321440   0.045668 0.185451  0.551207 0.747100 0.570486

CALIBRATION TABLE
                          model  prob_bin  events  mean_predicted_probability  observed_genuine_rate
          LOGIT_unweighted_full         0     269                    0.037256               0.096654
          LOGIT_unweighted_full         1     269                    0.106688     

In [ ]:
# =========================================================
# 14) BALANCED VS UNWEIGHTED FULL-MODEL COMPARISON
# =========================================================
balanced_full = pd.concat([
    balanced_predictions[
        balanced_predictions["model"] == "04_plus_volatility"
    ].assign(model="LOGIT_balanced_full"),
    rf_predictions[
        rf_predictions["model"] == "RF_04_plus_volatility"
    ].assign(model="RF_balanced_full"),
], ignore_index=True)

probability_comparison = pd.concat(
    [balanced_full, unweighted_predictions],
    ignore_index=True,
)

balanced_vs_unweighted = metrics_overall(probability_comparison)

print(balanced_vs_unweighted.to_string(index=False))


                          model  events  genuine_rate  roc_auc   pr_auc    brier  log_loss  mean_predicted_probability  top_quintile_events  top_quintile_genuine_rate  top_quintile_lift
            LOGIT_balanced_full  2687.0       0.32527 0.706785 0.528626 0.216854  0.630811                    0.458778                547.0                   0.564899           1.736710
          LOGIT_unweighted_full  2687.0       0.32527 0.702956 0.524159 0.196924  0.587118                    0.315190                547.0                   0.557587           1.714229
               RF_balanced_full  2687.0       0.32527 0.750583 0.575413 0.198837  0.582778                    0.446121                547.0                   0.625229           1.922184
RF_unweighted_C_plus_SC_no_MACD  2687.0       0.32527 0.747684 0.576291 0.184794  0.550082                    0.324119                547.0                   0.636197           1.955907
             RF_unweighted_full  2687.0       0.32527 0.747100 0.57048

In [ ]:
# =========================================================
# 15) TRANSPARENT LOGISTIC COEFFICIENT AUDIT
# =========================================================
# Interpretation only — NOT an OOS performance estimate.
# Fit the full transparent balanced specification after all OOS testing.

full_cols = MODEL_FEATURES["04_plus_volatility"]
X_full = features[full_cols].replace([np.inf, -np.inf], np.nan)
y_full = features["target"]

interp_pipe = make_logistic(class_weight="balanced")
interp_pipe.fit(X_full, y_full)

imputer = interp_pipe.named_steps["imputer"]
model = interp_pipe.named_steps["model"]

base_names = list(full_cols)
indicator_names = []

if hasattr(imputer, "indicator_") and imputer.indicator_ is not None:
    indicator_names = [
        f"MISSING__{full_cols[i]}"
        for i in imputer.indicator_.features_
    ]

all_names = base_names + indicator_names

coef_audit = pd.DataFrame({
    "feature": all_names,
    "standardized_logit_coefficient": model.coef_[0],
})

coef_audit["abs_coefficient"] = (
    coef_audit["standardized_logit_coefficient"].abs()
)

coef_audit = (
    coef_audit
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

print(coef_audit.head(40).to_string(index=False))


                                          feature  standardized_logit_coefficient  abs_coefficient
                           drawdown_from_126_high                        1.048342         1.048342
                        candidate_divergence_flag                       -0.736400         0.736400
                   candidate_divergence_active_63                        0.727751         0.727751
                        adverse_move_vs_incumbent                        0.723194         0.723194
                  candidate_dir_breakout_distance                        0.700276         0.700276
                                    ma_spread_pct                        0.666493         0.666493
                     candidate_dir_tsmom_distance                       -0.517407         0.517407
                        sc_osc_minus_upper_bb_pct                       -0.446831         0.446831
                      sc_candidate_dir_signal_pct                       -0.424310         0.424310
          

In [ ]:
# =========================================================
# 16) SAVE BOOK 04 OUTPUTS
# =========================================================
output_paths = {
    # Core predictions
    "balanced_logistic_predictions":
        V204 / "data" / "v2_04_balanced_logistic_predictions.parquet",
    "rf_ablation_predictions":
        V204 / "data" / "v2_04_rf_ablation_predictions.parquet",
    "unweighted_predictions":
        V204 / "data" / "v2_04_unweighted_predictions.parquet",

    # Main summaries
    "balanced_logistic_metrics":
        V204 / "results" / "v2_04_balanced_logistic_metrics.csv",
    "balanced_logistic_incremental":
        V204 / "results" / "v2_04_balanced_logistic_incremental.csv",
    "rf_ablation_metrics":
        V204 / "results" / "v2_04_rf_ablation_metrics.csv",
    "rf_ablation_incremental":
        V204 / "results" / "v2_04_rf_ablation_incremental.csv",
    "no_macd_comparison":
        V204 / "results" / "v2_04_no_macd_comparison.csv",

    # Robustness
    "metrics_by_asset_class":
        V204 / "results" / "v2_04_metrics_by_asset_class.csv",
    "metrics_by_direction":
        V204 / "results" / "v2_04_metrics_by_direction.csv",
    "traditional_assets_metrics":
        V204 / "results" / "v2_04_traditional_assets_metrics.csv",
    "yearly_metrics":
        V204 / "results" / "v2_04_yearly_metrics.csv",
    "yearly_stability":
        V204 / "results" / "v2_04_yearly_stability.csv",

    # Probability calibration
    "unweighted_metrics":
        V204 / "results" / "v2_04_unweighted_metrics.csv",
    "unweighted_calibration":
        V204 / "results" / "v2_04_unweighted_calibration.csv",
    "unweighted_calibration_error":
        V204 / "results" / "v2_04_unweighted_calibration_error.csv",
    "balanced_vs_unweighted":
        V204 / "results" / "v2_04_balanced_vs_unweighted.csv",

    # Interpretation / audit
    "logistic_coefficients":
        V204 / "results" / "v2_04_logistic_coefficient_audit.csv",
    "fold_audit":
        V204 / "results" / "v2_04_fold_audit.csv",
    "config":
        V204 / "config" / "v2_04_config.json",
}

balanced_predictions.to_parquet(
    output_paths["balanced_logistic_predictions"], index=False
)
rf_predictions.to_parquet(
    output_paths["rf_ablation_predictions"], index=False
)
unweighted_predictions.to_parquet(
    output_paths["unweighted_predictions"], index=False
)

balanced_metrics.to_csv(
    output_paths["balanced_logistic_metrics"], index=False
)
balanced_incremental.to_csv(
    output_paths["balanced_logistic_incremental"], index=False
)
rf_ablation_metrics.to_csv(
    output_paths["rf_ablation_metrics"], index=False
)
rf_incremental.to_csv(
    output_paths["rf_ablation_incremental"], index=False
)
no_macd_comparison.to_csv(
    output_paths["no_macd_comparison"], index=False
)

metrics_category.to_csv(
    output_paths["metrics_by_asset_class"], index=False
)
metrics_direction.to_csv(
    output_paths["metrics_by_direction"], index=False
)
traditional_metrics.to_csv(
    output_paths["traditional_assets_metrics"], index=False
)
yearly_metrics.to_csv(
    output_paths["yearly_metrics"], index=False
)
yearly_stability.to_csv(
    output_paths["yearly_stability"], index=False
)

unweighted_metrics.to_csv(
    output_paths["unweighted_metrics"], index=False
)
unweighted_calibration.to_csv(
    output_paths["unweighted_calibration"], index=False
)
unweighted_calibration_error.to_csv(
    output_paths["unweighted_calibration_error"], index=False
)
balanced_vs_unweighted.to_csv(
    output_paths["balanced_vs_unweighted"], index=False
)

coef_audit.to_csv(
    output_paths["logistic_coefficients"], index=False
)
fold_audit.to_csv(
    output_paths["fold_audit"], index=False
)

config_to_save = CONFIG.copy()
config_to_save["feature_families"] = MODEL_FEATURES
config_to_save["rf_feature_families"] = RF_FEATURES
config_to_save["source_book03"] = str(source_path)

with open(output_paths["config"], "w") as f:
    json.dump(config_to_save, f, indent=2, default=str)

print("Saved Book 04 outputs:")
for name, path in output_paths.items():
    print(f"  {name}: {path}")


Saved Book 04 outputs:
  balanced_logistic_predictions: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.04/data/v2_04_balanced_logistic_predictions.parquet
  rf_ablation_predictions: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.04/data/v2_04_rf_ablation_predictions.parquet
  unweighted_predictions: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.04/data/v2_04_unweighted_predictions.parquet
  balanced_logistic_metrics: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.04/results/v2_04_balanced_logistic_metrics.csv
  balanced_logistic_incremental: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.04/results/v2_04_balanced_logistic_incremental.csv
  rf_ablation_metrics: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.04/results/v2_04_rf_ablation_metrics.csv
  rf_ablation_increm

## Completion gate

Book 04 now includes the final targeted **no-MACD** ablation required before moving to latent-state modelling.

The decisive comparison is:

\[
RF(C) \quad \text{vs} \quad RF(C+M+SC) \quad \text{vs} \quad RF(C+SC)
\]

where:

- `C` = conventional transition geometry;
- `M` = MACD block;
- `SC` = SuperbCommand block.

This directly tests whether MACD is merely adding noise to an otherwise useful conventional + SuperbCommand transition model.

Before moving to the HMM stage, establish:

1. Does `RF(C+SC)` outperform `RF(C)` out of sample?
2. Does `RF(C+SC)` outperform `RF(C+M+SC)`?
3. Is any improvement visible in PR AUC, Brier/log loss and top-quintile lift as well as ROC AUC?
4. Does the result remain stronger for bull→bear than bear→bull transitions?
5. Is the result broad across asset classes?
6. Does the **unweighted** `RF(C+SC)` retain useful probability calibration?
7. Are results temporally stable?
8. Does excluding Bitcoin materially alter the conclusion?

If `C+SC` survives these tests, Book 04 can be frozen with a much cleaner conclusion: conventional geometry is the core transition engine, SuperbCommand is the principal incremental nonlinear feature family, while MACD and the separate volatility block do not earn standalone layers.

## Outputs to upload for final Book 04 analysis

### Required

1. `v2.04/results/v2_04_no_macd_comparison.csv`
2. `v2.04/results/v2_04_rf_ablation_incremental.csv`
3. `v2.04/results/v2_04_metrics_by_direction.csv`
4. `v2.04/results/v2_04_metrics_by_asset_class.csv`
5. `v2.04/results/v2_04_unweighted_metrics.csv`
6. `v2.04/results/v2_04_unweighted_calibration_error.csv`
7. `v2.04/results/v2_04_yearly_stability.csv`

### Preferred

8. `v2.04/results/v2_04_traditional_assets_metrics.csv`
9. `v2.04/results/v2_04_balanced_logistic_incremental.csv`
10. `v2.04/results/v2_04_balanced_vs_unweighted.csv`
11. `v2.04/results/v2_04_unweighted_calibration.csv`

The prediction Parquet files need not be uploaded initially.


## Research Outcome

Book 04 tested whether the candidate-time information developed in Book 03 can predict genuine versus failed trend transitions out of sample, and whether increasingly complex feature families provide information beyond conventional transition geometry.

The experiment uses expanding-window annual out-of-sample evaluation beginning in 2008. Each test year is predicted using earlier historical observations only; no random train/test split is used.

The feature-family falsification ladder tested:

1. conventional transition geometry;
2. + MACD;
3. + SuperbCommand;
4. + explicit volatility information;
5. conventional geometry + SuperbCommand without MACD.

Both transparent logistic models and nonlinear Random Forest models were evaluated. Balanced models were used primarily to study ranking/discrimination, while unweighted models were retained for probability-oriented interpretation.

The principal result is that **conventional transition geometry is strongly predictive, and SuperbCommand provides genuine incremental nonlinear information**. MACD does not earn a separate feature block, while an explicit standalone volatility block also fails to justify additional model complexity.

The preferred specification is therefore:

\[
\boxed{
P_t =
f_{\mathrm{RF}}
(
\text{Conventional Transition Geometry},
\text{SuperbCommand State}
)
}
\]

using an unweighted Random Forest.

Across 2,687 annual OOS transition candidates, the preferred probability model achieves approximately:

- **ROC AUC: 0.748**;
- **PR AUC: 0.576**;
- **Brier score: 0.185**;
- **log loss: 0.550**;
- actual genuine-transition rate: **32.53%**;
- mean predicted probability: **32.41%**.

Most importantly, the highest-ranked probability quintile contains genuine transitions at a rate of approximately:

\[
\boxed{63.62\%}
\]

versus the unconditional rate of:

\[
32.53\%.
\]

This represents approximately:

\[
\boxed{1.96\times}
\]

base-rate enrichment.

The signal is also directionally asymmetric. Conventional geometry is already particularly strong for bear-to-bull transitions, whereas SuperbCommand contributes substantially more incremental information when identifying deterioration of established bull trends. This argues against imposing artificial long/short symmetry in subsequent portfolio construction.

Performance is distributed across years and traditional asset classes rather than being driven by Bitcoin or a small number of episodes.

### Conclusion

Book 04 provides the central predictive result of the V2 research programme: **genuine trend transitions are meaningfully predictable before conventional confirmation**.

The resulting probability estimate is strong enough to justify proceeding from signal discovery to economic implementation. However, classification accuracy is not itself trading alpha. The next stage must establish whether increasing transition probability corresponds to economically exploitable forward-return distributions and whether anticipatory exposure improves a conventional trend-following portfolio after turnover, failed transitions and trading costs.

**Status: FROZEN. `RF_unweighted_C_plus_SC_no_MACD` is the V2 transition-probability benchmark.**